# 🎨 Sketch2Shape AI — Google Colab T4 Cloud Runner
### **TechZephyr 4.0 Hackathon (IIT Bhubaneswar)**

This self-contained notebook provides **one-click cloud acceleration** for Sketch2Shape AI on a free **NVIDIA T4 GPU runtime**.

**End-to-End Latency Target:** `< 25s` per 3D asset generation.

---

## 1. Verify NVIDIA T4 GPU Acceleration

In [ ]:
# Verify GPU allocation (Ensure Runtime -> Change runtime type -> T4 GPU is selected)
!nvidia-smi

## 2. Clone Repository or Navigate to Workspace

In [ ]:
import os

# Clone or pull latest repository
if not os.path.exists("ml_zephyr"):
    !git clone https://github.com/Banditapriyadarshini27/ml_zephyr.git
    %cd ml_zephyr
else:
    %cd ml_zephyr
    !git pull origin main

!pwd
!ls -la

## 3. Install Locked Dependencies & TripoSR Neural Engine

In [ ]:
# Install core system dependencies and neural libraries
!pip install --upgrade pip
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install diffusers>=0.26.0 transformers>=4.38.0 accelerate>=0.27.0
!pip install controlnet-aux rembg>=2.0.50 trimesh>=4.0.0 gradio>=4.20.0
!pip install einops omegaconf torchmcubes Pillow numpy opencv-python-headless
# Install TripoSR from official research repository
!pip install git+https://github.com/VAST-AI-Research/TripoSR.git

## 4. Pre-cache Model Weights (Optional Pre-warming for Sub-20s Latency)

In [ ]:
# Pre-download and cache model weights into huggingface cache
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
from tsr.system import TSR
import torch

print("Pre-warming ControlNet Scribble...")
cnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_scribble", torch_dtype=torch.float16)

print("Pre-warming Stable Diffusion v1.5...")
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=cnet,
    torch_dtype=torch.float16,
    safety_checker=None,
)

print("Pre-warming TripoSR...")
tsr_model = TSR.from_pretrained("stabilityai/TripoSR", config_name="config.yaml", weight_name="model.ckpt")

print("✅ All neural weights cached and ready for inference!")

## 5. Launch Gradio Web App with Public Tunnel (`share=True`)

Click the generated **`https://xxxx.gradio.live`** public link below to access the full interactive web application from any browser or device.

In [ ]:
import os
os.environ["GRADIO_SHARE"] = "1"

# Launch Gradio with public tunnel enabled
!python app.py